FASE 1
Importacion de librerias y cargar dataset

In [28]:
import pandas as pd
df = pd.read_csv('export_api_influencers.csv')
df.head(20)

,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,Bot_5,India,2023-05-06,AMATEUR_1
1,fan_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,Bot_11,India,2023-02-14,AMATEUR_1
3,Bot_1,India,2023-11-24,AMATEUR_1
4,Bot_36,India,2023-06-24,AMATEUR_1
5,Fan_4,Mexico,2023-05-29,GARNACHO
6,Bot_29,India,2023-08-18,AMATEUR_1
7,Bot_13,India,2023-08-05,AMATEUR_1
8,Bot_18,India,2023-04-30,AMATEUR_1
9,Fan_7,Mexico,2023-11-22,GARNACHO


Limpiar la estructura del codigo

In [29]:
df = df.dropna(subset=['Cuentas_Que_Sigue'])

df['Usuario'] = df['Usuario'].str.upper().str.strip()

df = df.drop_duplicates(subset=['Usuario'])

df.head(20)

,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,BOT_5,India,2023-05-06,AMATEUR_1
1,FAN_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,BOT_11,India,2023-02-14,AMATEUR_1
3,BOT_1,India,2023-11-24,AMATEUR_1
4,BOT_36,India,2023-06-24,AMATEUR_1
5,FAN_4,Mexico,2023-05-29,GARNACHO
6,BOT_29,India,2023-08-18,AMATEUR_1
7,BOT_13,India,2023-08-05,AMATEUR_1
8,BOT_18,India,2023-04-30,AMATEUR_1
9,FAN_7,Mexico,2023-11-22,GARNACHO


La explosion

In [30]:
# Separar las cuentas que sigue cada usuario
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.split(',')

# Expandir las cuentas en diferentes filas
df = df.explode('Cuentas_Que_Sigue')

df.head(20)

,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,BOT_5,India,2023-05-06,AMATEUR_1
1,FAN_6,Mexico,2023-07-14,GARNACHO
1,FAN_6,Mexico,2023-07-14,MESSI
2,BOT_11,India,2023-02-14,AMATEUR_1
3,BOT_1,India,2023-11-24,AMATEUR_1
4,BOT_36,India,2023-06-24,AMATEUR_1
5,FAN_4,Mexico,2023-05-29,GARNACHO
6,BOT_29,India,2023-08-18,AMATEUR_1
7,BOT_13,India,2023-08-05,AMATEUR_1
8,BOT_18,India,2023-04-30,AMATEUR_1


Purificacion post-explosion

In [31]:
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.upper().str.strip()

df = df[df['Cuentas_Que_Sigue'] != '']

df = df[df['Usuario'] != df['Cuentas_Que_Sigue']]

aristas = df[['Usuario', 'Cuentas_Que_Sigue']]

aristas.head(20)

,Usuario,Cuentas_Que_Sigue
0,BOT_5,AMATEUR_1
1,FAN_6,GARNACHO
1,FAN_6,MESSI
2,BOT_11,AMATEUR_1
3,BOT_1,AMATEUR_1
4,BOT_36,AMATEUR_1
5,FAN_4,GARNACHO
6,BOT_29,AMATEUR_1
7,BOT_13,AMATEUR_1
8,BOT_18,AMATEUR_1


FASE 2: El engaño de la popularidad

In [32]:
import networkx as nx

G = nx.from_pandas_edgelist(
    aristas,
    source='Usuario',
    target='Cuentas_Que_Sigue',
    create_using=nx.DiGraph()
)

print("Nodos:", G.number_of_nodes())
print("Aristas:", G.number_of_edges())

Nodos: 66
Aristas: 97


In [33]:
centralidad = nx.in_degree_centrality(G)

top3 = sorted(centralidad.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top 3 usuarios con mayor centralidad de entrada:")

for usuario, puntaje in top3:
    print(usuario, ":", puntaje)

Top 3 usuarios con mayor centralidad de entrada:
AMATEUR_1 : 0.7230769230769232
GARNACHO : 0.2
MESSI : 0.15384615384615385


FASE 3: La verdad topologica

In [34]:
pagerank = nx.pagerank(G)

top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)

print("Top 3 usuarios con mayor PageRank:")

for usuario, puntaje in top_pagerank[:3]:
    print(usuario, ":", puntaje)

Top 3 usuarios con mayor PageRank:
AMATEUR_1 : 0.1969406767931266
DE_PAUL : 0.08828460729048912
GARNACHO : 0.08428793226154431


In [35]:
G_no_dirigido = G.to_undirected()

clustering = nx.clustering(G_no_dirigido)

print("Clustering de AMATEUR_1:", clustering.get('AMATEUR_1', 0))
print("Clustering de GARNACHO:", clustering.get('GARNACHO', 0))

Clustering de AMATEUR_1: 0
Clustering de GARNACHO: 0.13333333333333333


Preguntas de veredicto final

1. ¿Quién compró bots y por qué?

El usuario que tiene muchos seguidores, pero un PageRank bajo, podría haber comprado bots, ya que su popularidad no representa necesariamente una influencia real.

2. ¿Quién es el influencer real y por qué?

El usuario con mayor PageRank es el influencer real, porque sus seguidores tienen importancia dentro de la red y sus conexiones le dan mayor autoridad.